In [1]:
import pandas as pd
import numpy as np
from binance_data_loader import BinanceDataLoader
from funding_arb_framework import (
    FundingDataBundle, FundingArbParams, FundingArbStrategy, BetaNeutralWeighting, FundingWalkForwardRunner
)
from itertools import product

In [2]:
loader = BinanceDataLoader(
    data_directory="/Users/chinjieheng/Documents/data/binance_1Hdata",
    funding_rate_directory="/Users/chinjieheng/Documents/data/binance_fundingrate_data",  # Add this line
    timeframe='1h',
    min_records=2160, # Ensure enough history for 90d lookback on 1h data
    min_volume=1e4,
    start_date="2024-09-01",
    end_date=None
)

price_hf = loader.get_price_matrix()
volume_hf = loader.get_volume_matrix(vol_30d=False) # Get raw volume matrix

#price_hf = price_hf.resample('1h').last()  # Resample to 1-hour intervals                                                                                                                                                         
returns_df_hf = price_hf.pct_change()                                                                                                                                               

# Prepare Daily Data (YOU MUST DO THIS)                                                                                                    
price_daily = price_hf.resample('D').last()      
                                                                                                                                                        
# Process Funding Data (Aggregate to Daily)                                                                                                                                                     
funding_long = loader.get_funding_long_form()                                                                                                                                                   
# Filter out 1h intervals if necessary, then sum daily

daily_funding = (                                                                                                                                                                               
    funding_long['fundingRate']                                                                                                                                                                 
    .unstack(level=0)                                                                                                                                                                           
    .sort_index()                                                                                                                                                                               
    .resample('D').sum(min_count=1)                                                                                                                                                             
)

Loading Binance data from /Users/chinjieheng/Documents/data/binance_1Hdata (timeframe=1h)...
Found 624 USDT trading pairs
Using a 720-bar rolling window for 30d volume checks
✓ BTCUSDT loaded successfully with 12588 records, avg volume: 714,743,488
Loaded 563 cryptocurrencies
Filtered 58 cryptocurrencies (insufficient data/volume)
Precomputing returns matrix (FAST numpy version)...
Building returns matrix (Memory Optimized)...
Matrix shape: (12588, 563)
Precomputed returns matrix shape: (12588, 563)
Date range: 2024-09-01 00:00:00 to 2026-02-07 11:00:00
Loading funding rate data from /Users/chinjieheng/Documents/data/binance_fundingrate_data...
Found 626 funding rate files
Loaded funding rates for 563 symbols
Building volume matrix (volume) for 563 tickers over 12588 dates...


/var/folders/4j/50_b76mn3qj360c47nzkyxdc0000gn/T/ipykernel_28589/2207489772.py:15: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_df_hf = price_hf.pct_change()


In [3]:
#The framework assumes price_df sets the "heartbeat" (frequency) of the backtest. If you passed hourly data as price_df, 
#the backtest would run hour-by-hour (which would be much slower and might not match your funding rate logic).  

# Create daily returns for the main backtest heartbeat
returns_daily = price_daily.pct_change()

bundle = FundingDataBundle(                                                                                                                                                                    
    price_df=price_daily,                                                                                                                                                                         
    funding_df=daily_funding,                                                                                                                                                                  
    returns_df=returns_daily,  # Daily returns for backtest heartbeat
    returns_df_hf=returns_df_hf,  # HF returns for beta calculation
    hf_window_multiplier=24, # 5-min bars: 24 hours * 12 bars/hour = 288 bars per day                                                                                                                                                                     
    min_hist_days=30,
    hf_resample_rule='D',
    volume_df=volume_hf # Pass volume data for filtering
)

/var/folders/4j/50_b76mn3qj360c47nzkyxdc0000gn/T/ipykernel_28589/2684428334.py:5: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_daily = price_daily.pct_change()


In [8]:
grid_choices = {                                                                                                                                                                               
    "ar_window": [30],                  # Lookback for AR(1) forecast                                                                                                                          
    "beta_window": [30],            # Lookback for short Beta calculation                                                                                                                        
    "portfolio_size_each_side": [5],    # Top 5 Long + Top 5 Short = 10 total positions at max                                                                                                                             
    "beta_limit": [1e-4],               # Tight beta neutrality                                                                                                                                
    "beta_type": ["btc"],#["adaptive"],   # Test vs Combined(BTC+ETH) or just BTC                                                                                                                
    "gross_exposure_limit": [0.3],      # individual coin cap (ya I'm lazy to change the variable name)                                                                                                                    
    "tc_bps": [5],                     # 5 bps transaction cost
    "min_positions": [6],       # Minimum active positions(long + short) (0 = off, !=0 force diversification)
    "min_weight": [0.05],     # Minimum absolute weight (we hard code to no leverage so total weight is 1.0, 0.05  means minimum 5% position size)
    "use_shrinkage": [True],          # Whether to use shrinkage in beta estimation
    "prior_beta_window": [90],      # Lookback for prior beta (target)  when using shrinkage
    "min_volume": [100000.0],        # Minimum 30-day ADV,
    "beta_neutral": [False]          # Whether to enforce beta neutrality
}          
                                                                                                                                                                                                                                                                                                                                                                                    
param_combinations = list(product(                                                                                                                                                             
    grid_choices["ar_window"],                                                                                                                                                                 
    grid_choices["beta_window"],                                                                                                                                                               
    grid_choices["portfolio_size_each_side"],                                                                                                                                                  
    grid_choices["beta_limit"],                                                                                                                                                                
    grid_choices["beta_type"],                                                                                                                                                                 
    grid_choices["gross_exposure_limit"],                                                                                                                                                      
    grid_choices["tc_bps"],
    grid_choices["min_positions"],
    grid_choices["min_weight"],
    grid_choices["use_shrinkage"],
    grid_choices["prior_beta_window"],
    grid_choices["min_volume"],
    grid_choices["beta_neutral"]
))               

params_grid = []                                                                                                                                                                               
for (ar_w, beta_w, p_size, b_lim, b_type, gross_lim, tc, min_pos, min_w, shrink, prior_beta_w, min_vol, beta_neu) in param_combinations:                                                                                                                
    params_grid.append(                                                                                                                                                                        
        FundingArbParams(                                                                                                                                                                      
            ar_window=ar_w,                                                                                                                                                                    
            beta_window=beta_w,                                                                                                                                                                
            portfolio_size_each_side=p_size,                                                                                                                                                   
            beta_limit=b_lim,                                                                                                                                                                  
            beta_type=b_type,                                                                                                                                                                  
            gross_exposure_limit=gross_lim,                                                                                                                                                    
            tc_bps=tc,
            min_positions=min_pos,
            min_weight=min_w,
            use_shrinkage=shrink,
            prior_beta_window=prior_beta_w,
            min_volume=min_vol,
            beta_neutral=beta_neu
        )                                                                                                                                                                                      
    )  

In [9]:

runner = FundingWalkForwardRunner(                                                                                                                                                              
    bundle=bundle,                                                                                                                                                                              
    params_grid=params_grid,                                                                                                                                                                    
    train_span=90,                                                                                                                                                                             
    test_span=90,                                                                                                                                                                               
    step_span=90,                                                                                                                                                                               
    score_mode="sharpe", # or "composite" 
    mode='expanding'                                                                                                                                                       
)    # Run                                                                                                                                                                                           
wf_df, oos_returns, oos_equity, oos_price_ret, oos_funding_ret, positions_df, detailed_df = runner.run()                                                                                                    
                                                                                                                                                                                                 
# Print DataFrame slice                                                                                                                                                                         
print(wf_df[[                                                                                                                                                                                   
     'iteration', 'train_start', 'train_end', 'test_start', 'test_end',                                                                                                                          
        'is_score', 'oos_score', 'is_sharpe', 'oos_sharpe'                                                                                                                                          
]])                                                                                                                                                                                             
                                                                                                                                                                                                 
# Helper function available in the module now                                                                                                                                                   
from funding_arb_framework import compute_sharpe                                                                                                                                                
                                                                                                                                                                                                 
print("Combined OOS Sharpe:", compute_sharpe(oos_returns, periods_per_year=365) if len(oos_returns) > 1 else float('nan'))                                                                      
                                                                                                                                                                                                 
# Generate Report                                                                                                                                                                               
report = runner.report(                                                                                                                                                                         
    wf_df=wf_df,                                                                                                                                                                                
    oos_returns=oos_returns,                                                                                                                                                                    
    oos_equity=oos_equity,
    oos_price_returns=oos_price_ret,                                                                                                                                               
    oos_funding_returns=oos_funding_ret,                                                                                                                                                                     
    plot=True,                                                                                                                                                                                  
    fig_dir="figures"                                                                                                                                                                           
)

# Save detailed records for comparison
if not detailed_df.empty:
    detailed_df.to_csv('detailed_records_framework_wf.csv', index=False)
    print(f"Saved {len(detailed_df)} detailed records to detailed_records_framework_wf.csv")

Iteration 1 (Expanding): Train [0:90], Test [90:179]
Iteration 2 (Expanding): Train [0:180], Test [180:269]
Iteration 3 (Expanding): Train [0:270], Test [270:359]
Iteration 4 (Expanding): Train [0:360], Test [360:449]
Iteration 5 (Expanding): Train [0:450], Test [450:524]
   iteration train_start  train_end test_start   test_end  is_score  \
0          1  2024-09-01 2024-11-30 2024-11-30 2025-02-27      -inf   
1          2  2024-09-01 2025-02-28 2025-02-28 2025-05-28  0.724808   
2          3  2024-09-01 2025-05-29 2025-05-29 2025-08-26  1.568611   
3          4  2024-09-01 2025-08-27 2025-08-27 2025-11-24  1.732861   
4          5  2024-09-01 2025-11-25 2025-11-25 2026-02-07  1.527452   

   oos_score  is_sharpe  oos_sharpe  
0   1.052903        NaN    1.052903  
1   2.535257   0.724808    2.535257  
2   3.314645   1.568611    3.314645  
3   0.597956   1.732861    0.597956  
4   2.129397   1.527452    2.129397  
Combined OOS Sharpe: 1.76560713454882
AGGREGATED OUT-OF-SAMPLE PERFORMAN

In [10]:
# Diagnostic: Verify fixes
print("=== FIXES VERIFICATION ===\n")

# 1. Check calendar-based dates
print("1. Calendar-based date handling:")
print(f"   First OOS date: {oos_returns.index[0]}")
print(f"   Last OOS date: {oos_returns.index[-1]}")
print(f"   Total OOS days: {len(oos_returns)}")

# 2. Check for flat days (missing data handling)
flat_days = (oos_returns == 0).sum()
print(f"\n2. Flat return days (missing data): {flat_days}")

# 3. Display return decomposition
print(f"\n3. Return decomposition:")
print(f"   Total Price Return: {oos_price_ret.sum()*100:.2f}%")
print(f"   Total Funding Return: {oos_funding_ret.sum()*100:.2f}%")
print(f"   Total Return: {(oos_equity.iloc[-1] - 1)*100:.2f}%")

# 4. Verify multiplicative vs additive consistency
additive_return = oos_returns.sum()
mult_return = oos_equity.iloc[-1] - 1
print(f"\n4. Additive vs Multiplicative:")
print(f"   Additive sum: {additive_return*100:.2f}%")
print(f"   Multiplicative: {mult_return*100:.2f}%")

# 5. Sample positions to verify trading
print(f"\n5. Sample positions (first 3 days with trades):")
if not positions_df.empty:
    sample = positions_df[positions_df['long_positions'] > 0].head(3)
    print(sample[['date', 'long_positions', 'short_positions', 'daily_return', 'price_return', 'funding_return']].to_string())

=== FIXES VERIFICATION ===

1. Calendar-based date handling:
   First OOS date: 2024-12-01 00:00:00
   Last OOS date: 2026-02-07 00:00:00
   Total OOS days: 430

2. Flat return days (missing data): 0

3. Return decomposition:
   Total Price Return: 21.85%
   Total Funding Return: 463.89%
   Total Return: 1162.13%

4. Additive vs Multiplicative:
   Additive sum: 464.03%
   Multiplicative: 1162.13%

5. Sample positions (first 3 days with trades):
        date  long_positions  short_positions  daily_return  price_return  funding_return
0 2024-12-01               3                3     -0.024637     -0.028127        0.003991
1 2024-12-02               2                4      0.037818      0.036288        0.001881
2 2024-12-03               1                5     -0.035734     -0.038241        0.002857


In [11]:
# =============================================================================
# GENERATE LATEST TARGET ALLOCATION
# =============================================================================
# This cell uses the best parameters found in the walk-forward process (or the last used set)
# to generate the target portfolio for the very next trading day. (but ofc u gotta get new data first just run the import notebook)

print("\n=== GENERATING LATEST TARGET ALLOCATION ===")

# 1. Identify Best Parameters
# In the report above, we saw the 'best' params. 
# For simplicity here, we'll take the parameters from the LAST iteration of the walk-forward
# which represents the most recent model configuration.
# Alternatively, you could hardcode the 'best' params observed in the report.

last_result = wf_df.iloc[-1]
best_params = last_result['best_params'] # Parameters used in the last confirmed step

print(f"Using parameters from Iteration {last_result['iteration']}:")
print(best_params)

# 2. Initialize Strategy with these params
strategy = FundingArbStrategy(best_params)
weighting = BetaNeutralWeighting()

# 3. Prepare Strategy
# The runner might have already prepared it, but to be safe and ensure we cover the absolute latest date:
print("Ensuring strategy signals are ready...")
strategy.prepare(bundle)

# 4. Get Latest Index
# We want the allocation for the 'Next Day' relative to our last data point.
idx = len(bundle.dates) - 1
last_date = pd.Timestamp(bundle.dates[idx])
print(f"Latest Data Date: {last_date.date()}")

# 5. Generate Signals & Weights
signals = strategy.signals(idx, bundle)

# Universe Mask (Price + Funding + Alpha/Beta validity)
p_curr = bundle.price_df.iloc[idx].to_numpy()
has_price = np.isfinite(p_curr)

if last_date in bundle.funding_df.index:
    f_curr = bundle.funding_df.loc[last_date].to_numpy()
    has_funding = np.isfinite(f_curr)
else:
    has_funding = np.zeros(len(bundle.tickers), dtype=bool)

base_mask = has_price & has_funding

alpha = signals["alpha"]
beta = signals["beta"]
valid_alpha = np.isfinite(alpha)
valid_beta = np.isfinite(beta)
uni_mask = base_mask & valid_alpha & valid_beta

# Apply Volume Filter (match framework)
if best_params.min_volume > 0 and bundle.volume_df is not None:
    vol_df = bundle.volume_df.copy()
    if not isinstance(vol_df.index, pd.DatetimeIndex):
        try:
            vol_df.index = pd.to_datetime(vol_df.index)
        except Exception:
            print("[WARNING] Volume DF index not datetime; skipping volume filter.")
            vol_df = None

    if vol_df is not None:
        avg_daily_vol = vol_df.rolling('30D', min_periods=1).sum() / 30.0
        avg_daily_vol = avg_daily_vol.reindex(bundle.price_df.index).fillna(0.0)

        if last_date in avg_daily_vol.index:
            current_vol = avg_daily_vol.loc[last_date].to_numpy()
            current_vol = np.nan_to_num(current_vol, 0.0)
            vol_mask = (current_vol >= best_params.min_volume)
            uni_mask = uni_mask & vol_mask
            print(f"Coins meeting volume criteria (> ${best_params.min_volume:,.0f}): {np.sum(vol_mask)}")
        else:
            print("[WARNING] Could not find volume data for current date, skipping volume filter.")

print(f"Coins eligible for trading (Price + Funding + Alpha/Beta + Vol): {np.sum(uni_mask)}")

weights = weighting.weights(idx, signals, bundle, uni_mask, best_params)

# 6. Output Table
print(f"\n{'='*40}")
print(f"TARGET ALLOCATION (Equity: 1.0 Unit)")
print(f"Date: {last_date.date()} (Close)")
print(f" { '='*40}")
print(f" { 'Symbol':<15} {'Side':<6} {'Weight':<10}")
print(f"{'-'*35}")

allocations = []
for i, w in enumerate(weights):
    if abs(w) > 1e-4:
        symbol = bundle.tickers[i]
        side = "LONG" if w > 0 else "SHORT"
        allocations.append({'Symbol': symbol, 'Side': side, 'Weight': w, 'Abs_W': abs(w)})

allocations.sort(key=lambda x: x['Abs_W'], reverse=True)

total_long = 0
total_short = 0

for item in allocations:
    print(f"{item['Symbol']:<15} {item['Side']:<6} {item['Weight']:<10.4f}")
    if item['Weight'] > 0:
        total_long += item['Weight']
    else:
        total_short += item['Weight']

print(f"{'-'*35}")
print(f"Total Long:  {total_long:.4f}")
print(f"Total Short: {total_short:.4f}")
print(f"Net Exposure: {total_long + total_short:.4f}")
print(f"Gross Exposure: {total_long - total_short:.4f}")
print(f" { '='*40}")


=== GENERATING LATEST TARGET ALLOCATION ===
Using parameters from Iteration 5:
FundingArbParams(ar_window=30, forecast_horizon=1, portfolio_size_each_side=5, beta_limit=0.0001, gross_exposure_limit=0.3, tc_bps=5, use_ar_model=True, beta_type='btc', beta_window=30, min_positions=6, min_weight=0.05, use_shrinkage=True, prior_beta_window=90, min_volume=100000.0, beta_neutral=False)
Ensuring strategy signals are ready...
Latest Data Date: 2026-02-07
Coins meeting volume criteria (> $100,000): 512
Coins eligible for trading (Price + Funding + Alpha/Beta + Vol): 497

TARGET ALLOCATION (Equity: 1.0 Unit)
Date: 2026-02-07 (Close)
 Symbol          Side   Weight    
-----------------------------------
LAUSDT          SHORT  -0.3000   
RIVERUSDT       LONG   0.3000    
NOMUSDT         LONG   0.2500    
ENSOUSDT        LONG   0.0500    
BERAUSDT        LONG   0.0500    
CLOUSDT         LONG   0.0500    
-----------------------------------
Total Long:  0.7000
Total Short: -0.3000
Net Exposure: 0.4